In [2]:
"""
Fractal Structure Analysis of Financial Time Series
=====================================================

Two analysis modules per ticker, saved under fractal_structure_results/:

  1. CUMULATIVE CHARACTERISTIC ANALYSIS  (inspired by Gaspard et al. 2000)
     -----------------------------------------------------------------------
     Financial analogue of the cumulative function F_k(θ) from the paper.
     4-panel layout:
       (a) Cumulative characteristic curves in ℂ for multiple q
       (b) D_H_geom(q) vs q²  — empirical box-count of F_q curves
                                 (geometric complexity of characteristic mapping)
       (c) MF-Lyapunov λ_MF and its components
       (d) D/λ decomposition: implied diffusion coefficient D̂

  2. MF-DFA  (Multifractal Detrended Fluctuation Analysis)
     -------------------------------------------------------
     Standard MF-DFA on log-returns.  6-panel layout:
       (a) Returns time series
       (b) F_q(s) log-log fluctuation functions
       (c) H(q) generalised Hurst exponent — annotated with H(q=2)
       (d) τ(q) scaling exponent
       (e) α(q) singularity strength — annotated with α₀
       (f) f(α) multifractal spectrum — annotated with Δα, λ_MF, and D_H

DIMENSION FRAMEWORK — TWO DISTINCT OBSERVABLES
------------------------------------------------
This code distinguishes two valid but different geometric objects:

  D_H  (spectral, theoretical)
  ─────────────────────────────
  The true Hausdorff dimension of the multifractal support, defined as:
      D_H = max_α f(α)
  Derived from the Legendre transform of τ(q) = q·H(q) − 1.
  This is the theoretically grounded quantity — stable, consistent, and
  embedded in the same MF-DFA framework as λ_MF and H(q).

  D_H_geom(q)  (empirical, q-dependent)
  ───────────────────────────────────────
  Box-counting dimension of the characteristic curve F_q in the complex
  plane, where F_q is the cumulative characteristic transform of returns
  at wavenumber q.  This is NOT the true support dimension — it measures
  the geometric complexity of the mapping ℝ → ℂ induced by q.  It varies
  with q precisely because the transform changes with q.

  Relationship:
      D_H_geom(q) ~ 1 + (D̂/λ_MF) · q²     [empirical scaling, panel b]
  where D̂ is the implied diffusion coefficient.

  Why keep D_H_geom(q)?
  ─────────────────────
  It captures a different physical story: how the geometric complexity of
  the characteristic mapping responds to the "frequency" q.  In the Gaspard
  et al. framework this is the analogue of the fractal dimension of the
  chaotic repeller as a function of the wavenumber k.  Keeping it alongside
  the spectral D_H allows us to distinguish intrinsic multifractal geometry
  from induced characteristic-map complexity.

LYAPUNOV ESTIMATION — MF-SPECTRUM APPROACH
-------------------------------------------
In Gaspard et al. the positive Lyapunov exponent λ is defined via
Ruelle's topological pressure:

    P(β) = lim_{t→∞} (1/t) ln ⟨|Λ_t|^{1−β}⟩
    λ = −dP/dβ |_{β=1}

The multifractal spectrum f(α) is the Legendre transform of τ(q) = q·H(q) − 1,
which is itself the financial analogue of the topological pressure.  So both
λ and f(α) are rooted in the same underlying object — they are not independent.

We estimate λ directly from the MF-DFA output:

    λ_MF = H(q=2) · σ · (1 + Δα)

  • H(q=2)  — generalised Hurst at q=2, the standard Hurst exponent.
              Controls the scaling of typical (second-moment) fluctuations.
              Analogous to the mean stretching rate of trajectories.

  • σ       — return standard deviation.  Grounds λ in the actual scale
              of fluctuations (units: per day).

  • (1 + Δα) — multifractality correction.  Δα = α_max − α_min is the
              width of the singularity spectrum.  A wider spectrum means
              more heterogeneity in local Hölder exponents — some regions
              are far more singular than others — which is the financial
              analogue of trajectory-divergence heterogeneity in phase space.
              For a pure Brownian motion Δα → 0 and the correction → 1.

This keeps λ_MF internally consistent with D_H: both are derived from
the same MF-DFA computation, so the D/λ slope on panel (b) is
self-consistent rather than mixing two independent estimators.
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.signal import savgol_filter
import os
import warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
TICKERS   = ["AAPL", "MSFT", "GOOG", "AMZN", "NVDA", "JPM", "XOM", "BTC-USD"]
START     = "2010-01-01"
END       = "2026-03-01"
OUT_DIR   = "fractal_structure_results"

# q values for cumulative characteristic (wavenumber analogue)
Q_CHAR         = np.array([0, 1, 5, 10, 20, 60, 120, 200, 300], dtype=float)
Q_CHAR_DISPLAY = [0, 1, 5, 10, 20, 60, 200, 300]

# q values for MF-DFA
Q_HURST = np.arange(-30, 31, 5, dtype=float)

# MF-DFA scale parameters
SCALE_MIN = 20
SCALE_MAX = 800
N_SCALES  = 30
POLY_ORD  = 1
OVERLAP   = 0.75

# Palette for F_q(s) lines
CMAP_LINES = [
    "#c0392b", "#2980b9", "#27ae60", "#8e44ad",
    "#d35400", "#16a085", "#2c3e50", "#e67e22",
    "#1abc9c", "#e74c3c", "#3498db",
]

os.makedirs(OUT_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# STYLE
# ─────────────────────────────────────────────────────────────────────────────
PLT_STYLE = dict(
    fig_fc    = "#ffffff",
    ax_fc     = "#ffffff",
    text      = "#1a1a1a",
    grid      = "#e5e5e5",
    spine     = "#cccccc",
    accent    = "#1a1a2e",
    line_main = "#1f4e8c",
    line_alt  = "#c0392b",
    line_grn  = "#1a7a4a",
)

def _style(fig, axes_flat):
    fig.patch.set_facecolor(PLT_STYLE["fig_fc"])
    for ax in axes_flat:
        ax.set_facecolor(PLT_STYLE["ax_fc"])
        ax.tick_params(colors=PLT_STYLE["text"], which="both",
                       direction="in", top=True, right=True, labelsize=8)
        ax.xaxis.label.set_color(PLT_STYLE["text"])
        ax.yaxis.label.set_color(PLT_STYLE["text"])
        ax.title.set_color(PLT_STYLE["accent"])
        for sp in ax.spines.values():
            sp.set_edgecolor(PLT_STYLE["spine"])
        ax.grid(True, color=PLT_STYLE["grid"], linewidth=0.5,
                linestyle="--", alpha=0.9)

def _legend(ax, **kw):
    return ax.legend(framealpha=0.9, facecolor="white",
                     edgecolor=PLT_STYLE["spine"],
                     labelcolor=PLT_STYLE["text"], **kw)


# ─────────────────────────────────────────────────────────────────────────────
# DIRECTORIES
# ─────────────────────────────────────────────────────────────────────────────
def get_dirs(ticker):
    t    = ticker.replace("=", "_")
    base = os.path.join(OUT_DIR, t)
    char = os.path.join(base, "characteristic_curves")
    mfd  = os.path.join(base, "mfdfa")
    os.makedirs(char, exist_ok=True)
    os.makedirs(mfd,  exist_ok=True)
    return char, mfd


# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────
def get_data(ticker):
    df = yf.download(ticker, start=START, end=END, progress=False)
    if df is None or len(df) == 0:
        return None, None
    try:
        if hasattr(df.columns, "levels"):
            px = df["Close"][ticker].dropna().values
        else:
            px = df["Close"].dropna().values
    except Exception:
        px = df["Close"].dropna().values

    px = px.ravel().astype(float)
    px = px[px > 0]
    if len(px) < 200:
        return None, None

    logp = np.log(px)
    ret  = np.diff(logp)
    ret  = ret[np.isfinite(ret)]
    if len(ret) < 200:
        return None, None
    return logp, ret


# ─────────────────────────────────────────────────────────────────────────────
# MF-DFA  (computed first — λ_MF and D_H depend on its output)
# ─────────────────────────────────────────────────────────────────────────────
def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)


def mfdfa(x, q_vals, scale_min=SCALE_MIN, scale_max=SCALE_MAX,
          n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP):
    """MF-DFA on log-returns with overlapping segments."""
    x  = x[np.isfinite(x)]
    Y  = np.cumsum(x - x.mean())
    N  = len(Y)

    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    Fqs = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            continue
        t   = np.arange(s)
        rms = []
        for st in starts:
            seg  = Y[st: st + s]
            coef = np.polyfit(t, seg, poly_ord)
            r_v  = np.sqrt(np.mean((seg - np.polyval(coef, t)) ** 2))
            if r_v > 0:
                rms.append(r_v)
        rms = np.asarray(rms)
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms ** 2)))
            else:
                Fqs[j, i] = (np.mean(rms ** q)) ** (1.0 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]

    return scales, Fqs, hq


def multifractal_spectrum(q_vals, hq):
    """τ(q), α(q), f(α) from H(q)."""
    tau_q = q_vals * hq - 1.0
    wl    = min(5, len(tau_q) if len(tau_q) % 2 == 1 else len(tau_q) - 1)
    tau_s = (savgol_filter(tau_q, window_length=wl, polyorder=2)
             if len(tau_q) >= 5 else tau_q)
    alpha   = np.gradient(tau_s, q_vals)
    D_alpha = q_vals * alpha - tau_s
    return tau_q, alpha, D_alpha


# ─────────────────────────────────────────────────────────────────────────────
# SPECTRAL HAUSDORFF DIMENSION  (replaces box-counting as primary D_H)
# ─────────────────────────────────────────────────────────────────────────────
def hausdorff_from_spectrum(D_alpha):
    """
    True Hausdorff dimension of the multifractal support:
        D_H = max_α f(α)

    This is the theoretically grounded value, derived directly from the
    multifractal spectrum.  In the full multifractal formalism, f(α) gives
    the Hausdorff dimension of the set of points with local Hölder exponent α.
    The peak of f(α) equals the dimension of the support — i.e. the whole set.

    For a monofractal (e.g. pure Brownian motion), f(α) collapses to a point
    and D_H = 1.  For richer multifractals the peak typically exceeds 1 and
    the curve broadens.

    Returns:
        D_H : float  — Hausdorff dimension of the support (∈ [1, 2] for curves)
    """
    valid = np.isfinite(D_alpha)
    if valid.sum() < 2:
        return 1.0
    return float(np.clip(np.nanmax(D_alpha[valid]), 1.0, 2.0))


# ─────────────────────────────────────────────────────────────────────────────
# EMPIRICAL BOX-COUNTING  (secondary — geometric complexity of F_q mapping)
# ─────────────────────────────────────────────────────────────────────────────
def hausdorff_boxcount(F, n_eps=15):
    """
    Box-counting dimension of the complex characteristic curve F_q.

    This measures the geometric complexity of the characteristic mapping
    ℝ → ℂ induced by wavenumber q.  It is NOT the true support dimension —
    it is q-dependent precisely because the transform changes with q.

    Used only for the empirical D_H_geom(q) vs q² plot (panel b of
    cumulative characteristic figure), which captures how geometric
    complexity scales with q — analogous to the fractal dimension of the
    chaotic repeller as a function of wavenumber k in Gaspard et al.
    """
    x, y = F.real, F.imag
    if len(x) < 10:
        return 1.0
    eps    = np.logspace(-1.0, -2.5, n_eps)
    counts = [len(set(zip((x / e).astype(int), (y / e).astype(int))))
              for e in eps]
    counts = np.array(counts, dtype=float)
    valid  = counts > 0
    if valid.sum() < 3:
        return 1.0
    slope, _ = np.polyfit(np.log(eps[valid]), np.log(counts[valid]), 1)
    return float(np.clip(-slope, 1.0, 2.0))


# ─────────────────────────────────────────────────────────────────────────────
# MF-DERIVED LYAPUNOV EXPONENT
# ─────────────────────────────────────────────────────────────────────────────
def lyapunov_mf(ret, q_vals, hq, alpha, D_alpha):
    """
    Estimate the Lyapunov exponent from the multifractal spectrum.

    Formula:
        λ_MF = H(q=2) · σ · (1 + Δα)

    Components:
      H(q=2)  — standard Hurst exponent from MF-DFA at q=2.
                 Controls scaling of second-moment fluctuations.
                 Analogous to the mean stretching rate in phase space.

      σ       — return standard deviation (daily scale).
                 Grounds λ_MF in the physical scale of fluctuations.

      Δα      — singularity spectrum width = α_max − α_min.
                 Measures heterogeneity of local Hölder exponents.
                 Δα → 0 for monofractal (pure BM); larger Δα means
                 more intermittency and stronger effective divergence.
                 Acts as a multifractality correction to H(q=2)·σ.

    Also computes D_H (spectral) = max f(α), the true Hausdorff dimension
    of the support, which is used as the primary geometric quantity in all
    plots and annotations.

    Returns dict with full decomposition for plotting/annotation.
    """
    sigma = float(np.std(ret))

    # H at q=2
    q2_idx = np.argmin(np.abs(q_vals - 2.0))
    H2     = float(hq[q2_idx]) if np.isfinite(hq[q2_idx]) else 0.5

    # spectrum quantities
    valid_s = np.isfinite(alpha) & np.isfinite(D_alpha)
    if valid_s.sum() >= 2:
        delta_alpha = float(alpha[valid_s].max() - alpha[valid_s].min())
        alpha0      = float(alpha[np.argmin(np.abs(q_vals))])   # at q=0
        f_alpha0    = float(D_alpha[np.argmin(np.abs(q_vals))]) # should ≈ 1
    else:
        delta_alpha = 0.0
        alpha0      = H2
        f_alpha0    = 1.0

    lam_mf = H2 * sigma * (1.0 + delta_alpha)
    lam_mf = float(np.clip(lam_mf, 1e-6, None))

    # ── Spectral D_H: the theoretically grounded Hausdorff dimension ──────────
    # D_H = max_α f(α)  — peak of the multifractal spectrum.
    # This is consistent with λ_MF since both come from the same MF-DFA output.
    D_H_spectral = hausdorff_from_spectrum(D_alpha)

    return dict(
        lam_mf       = lam_mf,
        H2           = H2,
        sigma        = sigma,
        delta_alpha  = delta_alpha,
        alpha0       = alpha0,
        f_alpha0     = f_alpha0,
        D_H_spectral = D_H_spectral,   # primary geometric quantity
    )


# ─────────────────────────────────────────────────────────────────────────────
# CUMULATIVE CHARACTERISTIC ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
def cumulative_characteristic(r, q, n_grid=2000):
    r = np.asarray(r, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 10:
        return np.linspace(0, 1, n_grid) + 0j

    # subsample to n_grid points but PRESERVE time order
    if len(r) > n_grid:
        idx = np.linspace(0, len(r) - 1, n_grid).astype(int)
        r_used = r[idx]
    else:
        r_used = r

    # cumulative sum along TIME axis — this is what generates fractal structure
    F = np.cumsum(np.exp(1j * q * r_used))

    # normalise to unit bounding box for comparability across q
    d = F[-1] - F[0]
    if np.abs(d) < 1e-12:
        return F / (np.max(np.abs(F)) + 1e-12)
    return (F - F[0]) / d


# ─────────────────────────────────────────────────────────────────────────────
# PLOT A — CUMULATIVE CHARACTERISTIC  (2×2 panels)
# ─────────────────────────────────────────────────────────────────────────────
def plot_characteristic(ticker, ret, lam_dict, char_dir):
    """
    4-panel figure.

    Panel (b) now shows D_H_geom(q) — the empirical box-count of the
    characteristic curves F_q — clearly labelled as the induced geometric
    complexity of the mapping, distinct from the true spectral D_H which
    is reported as an annotation and derived from max f(α).

    The q²-scaling of D_H_geom(q) follows:
        D_H_geom(q) ≈ 1 + (D̂ / λ_MF) · q²
    where D̂ is the implied diffusion coefficient — this is the Gaspard
    et al. analogue, now cleanly separated from the intrinsic dimension.
    """
    # Empirical box-count — geometric complexity of mapping (q-dependent)
    DH_geom = np.array([hausdorff_boxcount(cumulative_characteristic(ret, q))
                        for q in Q_CHAR])
    q2      = Q_CHAR ** 2
    valid_m = np.isfinite(DH_geom) & (q2 > 0)

    slope_fit, intercept_fit = (np.polyfit(q2[valid_m], DH_geom[valid_m], 1)
                                if valid_m.sum() >= 2 else (np.nan, 1.0))

    lam_mf       = lam_dict["lam_mf"]
    H2           = lam_dict["H2"]
    sigma        = lam_dict["sigma"]
    delta_alpha  = lam_dict["delta_alpha"]
    D_H_spectral = lam_dict["D_H_spectral"]   # primary: from max f(α)

    # Implied diffusion coefficient from D_H_geom slope
    D_implied = slope_fit * lam_mf if np.isfinite(slope_fit) else np.nan

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"Cumulative Characteristic Analysis  ·  {ticker}",
        fontsize=13, fontweight="bold", color=PLT_STYLE["accent"], y=1.01,
    )
    _style(fig, axes.ravel())

    # ── (a) curves in ℂ ──────────────────────────────────────────────────────
    ax = axes[0, 0]
    for i, q in enumerate(Q_CHAR):
        c = plt.cm.magma_r(i / len(Q_CHAR))
        F  = cumulative_characteristic(ret, q)
        lw = 1.0 if q > 20 else 1.6
        ax.plot(F.real, F.imag, linewidth=lw, color=c,
                label=f"$q={int(q)}$", alpha=0.88)
    ax.set_xlabel("Re $F_q$", fontsize=10)
    ax.set_ylabel("Im $F_q$", fontsize=10)
    ax.set_title("(a)  Cumulative characteristic curves", fontsize=10)
    _legend(ax, fontsize=8)
    ax.set_aspect("equal", adjustable="datalim")
    # annotate spectral D_H in corner
    ax.text(0.03, 0.97,
            fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="left", va="top", fontsize=8,
            color=PLT_STYLE["line_grn"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    # ── (b) D_H_geom(q) vs q²  ───────────────────────────────────────────────
    # This is the EMPIRICAL, q-dependent box-count dimension — NOT the true D_H.
    # It measures geometric complexity of the characteristic mapping at each q.
    ax = axes[0, 1]
    ax.scatter(q2, DH_geom, color=PLT_STYLE["line_main"], s=60, zorder=5,
               edgecolors="#aaaaaa", linewidths=0.5,
               label=r"$D_H^{\rm geom}(q)$  (box-count of $F_q$)")
    if np.isfinite(slope_fit):
        q2_line = np.linspace(0, q2.max() * 1.06, 200)
        ax.plot(q2_line, intercept_fit + slope_fit * q2_line,
                color=PLT_STYLE["line_alt"], linewidth=1.5, linestyle="--",
                label=fr"fit  slope$={slope_fit:.2e}$")
        if np.isfinite(D_implied):
            ax.plot(q2_line, 1.0 + (D_implied / lam_mf) * q2_line,
                    color=PLT_STYLE["line_grn"], linewidth=1.2, linestyle=":",
                    label=fr"$\hat{{D}}/\lambda_{{MF}}={D_implied/lam_mf:.2e}$")
    # Draw horizontal line at spectral D_H — the true intrinsic dimension
    ax.axhline(D_H_spectral, color=PLT_STYLE["line_grn"], linewidth=1.2,
               linestyle="-.",
               label=fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$")
    ax.axhline(1.0, color="#aaaaaa", linewidth=0.8, linestyle=":")
    ax.set_xlabel("$q^2$",    fontsize=10)
    ax.set_ylabel(r"$D_H^{\rm geom}(q)$", fontsize=10)
    ax.set_title(
        r"(b)  $D_H^{\rm geom}(q)$ vs $q^2$  — induced geometric complexity"
        "\n" + r"$D_H^{\rm geom}(q)\approx 1 + (\hat{\mathcal{D}}/\lambda)\,q^2$",
        fontsize=9)
    _legend(ax, fontsize=7)
    ax.text(0.97, 0.12,
            fr"$\lambda_{{MF}} = {lam_mf:.4f}$",
            transform=ax.transAxes, ha="right", fontsize=9,
            color=PLT_STYLE["line_main"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    # ── (c) λ_MF component breakdown ─────────────────────────────────────────
    ax = axes[1, 0]
    labels  = ["$H(q{=}2)$", r"$\sigma$", r"$1+\Delta\alpha$", r"$\lambda_{MF}$"]
    values  = [H2, sigma, 1.0 + delta_alpha, lam_mf]
    colours = [PLT_STYLE["line_main"], PLT_STYLE["line_grn"],
               "#e67e22", PLT_STYLE["line_alt"]]
    bars = ax.bar(labels, values, color=colours, edgecolor="#cccccc",
                  linewidth=0.8, alpha=0.85, width=0.55)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values) * 0.02,
                f"{val:.4f}", ha="center", va="bottom",
                fontsize=8, color=PLT_STYLE["text"])
    ax.set_ylabel("Value", fontsize=10)
    ax.set_title(
        r"(c)  $\lambda_{MF} = H(q{=}2)\cdot\sigma\cdot(1+\Delta\alpha)$",
        fontsize=10)
    ax.set_ylim(0, max(values) * 1.18)
    ax.text(0.97, 0.97,
            fr"$\lambda_{{MF}} = {H2:.3f}\times{sigma:.4f}\times{1+delta_alpha:.3f}$"
            f"\n$= {lam_mf:.5f}$"
            f"\n$D_H = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            color=PLT_STYLE["text"],
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    # ── (d) Implied diffusion coefficient D̂ ──────────────────────────────────
    # D_H_geom - 1 vs q²/λ_MF  — should be linear with slope = D̂
    # Uses the empirical D_H_geom(q), not the spectral D_H.
    ax = axes[1, 1]
    dh_excess = DH_geom - 1.0
    q2_scaled = q2 / lam_mf
    ax.scatter(q2_scaled[valid_m], dh_excess[valid_m],
               color=PLT_STYLE["line_main"], s=60, zorder=5,
               edgecolors="#aaaaaa", linewidths=0.5,
               label=r"$(D_H^{\rm geom} - 1)$ vs $q^2/\lambda_{MF}$")
    if np.isfinite(D_implied) and valid_m.sum() >= 2:
        x_line = np.linspace(0, q2_scaled[valid_m].max() * 1.06, 200)
        ax.plot(x_line, D_implied * x_line,
                color=PLT_STYLE["line_alt"], linewidth=1.5, linestyle="--",
                label=fr"slope $= \hat{{D}} = {D_implied:.4f}$")
    ax.axhline(0, color="#aaaaaa", linewidth=0.8, linestyle=":")
    # annotate spectral D_H - 1 as reference
    ax.axhline(D_H_spectral - 1.0, color=PLT_STYLE["line_grn"],
               linewidth=1.0, linestyle="-.",
               label=fr"$D_H - 1 = {D_H_spectral-1:.4f}$  (spectral)")
    ax.set_xlabel(r"$q^2 / \lambda_{MF}$",  fontsize=10)
    ax.set_ylabel(r"$D_H^{\rm geom} - 1$",  fontsize=10)
    ax.set_title(r"(d)  Implied diffusion coefficient $\hat{\mathcal{D}}$",
                 fontsize=10)
    _legend(ax, fontsize=7)
    if np.isfinite(D_implied):
        ax.text(0.97, 0.12,
                fr"$\hat{{\mathcal{{D}}}} = {D_implied:.5f}$",
                transform=ax.transAxes, ha="right", fontsize=9,
                color=PLT_STYLE["line_alt"],
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                          edgecolor=PLT_STYLE["spine"], alpha=0.85))

    plt.tight_layout()
    fname = os.path.join(char_dir, f"{ticker.replace('=','_')}_characteristic.png")
    plt.savefig(fname, dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"    saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT B — MF-DFA 6-panel
# ─────────────────────────────────────────────────────────────────────────────
def plot_mfdfa(ticker, ret, scales, Fqs, hq, q_vals,
               tau_q, alpha, D_alpha, lam_dict, mfd_dir):
    valid   = np.isfinite(hq)
    valid_s = np.isfinite(alpha) & np.isfinite(D_alpha)
    colors  = (CMAP_LINES * 5)[: len(q_vals)]

    lam_mf       = lam_dict["lam_mf"]
    H2           = lam_dict["H2"]
    delta_alpha  = lam_dict["delta_alpha"]
    alpha0       = lam_dict["alpha0"]
    D_H_spectral = lam_dict["D_H_spectral"]   # primary geometric quantity

    fig, axs = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(
        f"Multifractal DFA  ·  {ticker}  ·  log-returns",
        fontsize=13, fontweight="bold", color=PLT_STYLE["accent"], y=1.01,
    )
    _style(fig, axs.ravel())

    # (a) Returns
    ax = axs[0, 0]
    ax.plot(ret, color=PLT_STYLE["line_main"], linewidth=0.55, alpha=0.9)
    ax.set_xlabel("$t$  (trading days)", fontsize=10)
    ax.set_ylabel("Log-return",           fontsize=10)
    ax.set_title("(a)  Log-return series", fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3g"))

    # (b) F_q(s)
    ax = axs[0, 1]
    legend_handles = []
    for idx_q, (q, c) in enumerate(zip(q_vals, colors)):
        row  = Fqs[idx_q]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() < 2:
            continue
        h, = ax.loglog(scales[mask], row[mask], "o-",
                       color=c, markersize=2.8, linewidth=0.85, alpha=0.8)
        legend_handles.append((h, f"$q={int(q)}$"))
    ax.set_xlabel("Scale $s$", fontsize=10)
    ax.set_ylabel("$F_q(s)$",  fontsize=10)
    ax.set_title("(b)  Fluctuation functions $F_q(s)$", fontsize=10)
    step  = max(1, len(legend_handles) // 7)
    shown = legend_handles[::step]
    if shown:
        _legend(ax, handles=[h for h, _ in shown],
                labels=[l for _, l in shown],
                fontsize=7, ncol=1, loc="best",
                handlelength=1.2, handletextpad=0.4)

    # (c) H(q)  — annotate H(q=2) and D_H (spectral)
    ax = axs[0, 2]
    ax.plot(q_vals[valid], hq[valid], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    ax.axhline(0.5, color="#aaaaaa", lw=0.9, ls="--", alpha=0.9, label="$H=0.5$")
    ax.axhline(1.0, color=PLT_STYLE["line_alt"], lw=0.9, ls=":", alpha=0.9,
               label="$H=1.0$")
    q2_idx = np.argmin(np.abs(q_vals - 2.0))
    if np.isfinite(hq[q2_idx]):
        ax.plot(q_vals[q2_idx], hq[q2_idx], "o",
                color=PLT_STYLE["line_alt"], markersize=7, zorder=6,
                label=f"$H(q{{=}}2)={H2:.3f}$")
    ax.set_xlabel("$q$",    fontsize=10)
    ax.set_ylabel("$H(q)$", fontsize=10)
    ax.set_title("(c)  Generalised Hurst exponent $H(q)$", fontsize=10)
    _legend(ax, fontsize=8)
    # note D_H in corner
    ax.text(0.03, 0.05,
            fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="left", va="bottom", fontsize=8,
            color=PLT_STYLE["line_grn"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    # (d) τ(q)
    ax = axs[1, 0]
    ax.plot(q_vals[valid], tau_q[valid], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    ax.axhline(0, color="#aaaaaa", lw=0.8, ls="--", alpha=0.8)
    ax.set_xlabel("$q$",        fontsize=10)
    ax.set_ylabel(r"$\tau(q)$", fontsize=10)
    ax.set_title(r"(d)  Scaling exponent $\tau(q)$", fontsize=10)

    # (e) α(q)  — annotate α₀
    ax = axs[1, 1]
    ax.plot(q_vals[valid_s], alpha[valid_s], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    q0_idx = np.argmin(np.abs(q_vals))
    if np.isfinite(alpha[q0_idx]):
        ax.plot(q_vals[q0_idx], alpha[q0_idx], "o",
                color=PLT_STYLE["line_alt"], markersize=7, zorder=6,
                label=fr"$\alpha_0={alpha0:.3f}$")
    ax.set_xlabel("$q$",          fontsize=10)
    ax.set_ylabel(r"$\alpha(q)$", fontsize=10)
    ax.set_title(r"(e)  Singularity strength $\alpha(q)$", fontsize=10)
    _legend(ax, fontsize=8)

    # (f) f(α)  — primary home for D_H annotation
    #             peak of this curve IS D_H by definition
    ax = axs[1, 2]
    ax.plot(alpha[valid_s], D_alpha[valid_s], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    if np.isfinite(alpha[q0_idx]) and np.isfinite(D_alpha[q0_idx]):
        ax.plot(alpha[q0_idx], D_alpha[q0_idx], "o",
                color=PLT_STYLE["line_alt"], markersize=7, zorder=6,
                label=f"$q=0$  ($\\alpha_0={alpha0:.3f}$)")
    # mark the peak — this is D_H
    if valid_s.sum() >= 2:
        peak_idx   = np.nanargmax(D_alpha[valid_s])
        alpha_peak = alpha[valid_s][peak_idx]
        f_peak     = D_alpha[valid_s][peak_idx]
        ax.plot(alpha_peak, f_peak, "^",
                color=PLT_STYLE["line_grn"], markersize=9, zorder=7,
                label=fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$")
        ax.annotate(fr"$D_H={D_H_spectral:.4f}$",
                    xy=(alpha_peak, f_peak),
                    xytext=(alpha_peak + 0.02, f_peak - 0.06),
                    fontsize=8, color=PLT_STYLE["line_grn"],
                    arrowprops=dict(arrowstyle="->", color=PLT_STYLE["line_grn"],
                                   lw=0.9))
        # draw Δα span at the base
        a_min  = alpha[valid_s].min()
        a_max  = alpha[valid_s].max()
        y_span = D_alpha[valid_s].min() - 0.05
        ax.annotate("", xy=(a_max, y_span), xytext=(a_min, y_span),
                    arrowprops=dict(arrowstyle="<->", color="#888888", lw=1.2))
        ax.text((a_min + a_max) / 2, y_span - 0.04,
                fr"$\Delta\alpha={delta_alpha:.3f}$",
                ha="center", fontsize=8, color=PLT_STYLE["text"])
    ax.set_xlabel(r"$\alpha$",    fontsize=10)
    ax.set_ylabel(r"$f(\alpha)$", fontsize=10)
    ax.set_title(r"(f)  Multifractal spectrum $f(\alpha)$"
                 "\n" + r"$D_H = \max_\alpha f(\alpha)$  (no box-counting needed)",
                 fontsize=9)
    _legend(ax, fontsize=8)
    # λ_MF and D_H summary box
    ax.text(0.97, 0.30,
            fr"$\lambda_{{MF}} = {lam_mf:.5f}$"
            "\n"
            fr"$= H_2\cdot\sigma\cdot(1+\Delta\alpha)$"
            "\n"
            fr"$D_H = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            color=PLT_STYLE["text"],
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.9))

    plt.tight_layout()
    fname = os.path.join(mfd_dir, f"{ticker.replace('=','_')}_mfdfa.png")
    plt.savefig(fname, dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"    saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Ticker: {ticker}")

    char_dir, mfd_dir = get_dirs(ticker)
    logp, ret = get_data(ticker)

    if ret is None:
        print("  Skipping — insufficient data.")
        continue

    N = len(ret)
    print(f"  N = {N} returns")

    # ── MF-DFA first (λ_MF and D_H both depend on its output) ────────────────
    scale_max_eff = min(SCALE_MAX, N // 4)
    scale_min_eff = max(SCALE_MIN, POLY_ORD + 2)
    print(f"  → MF-DFA  (scales [{scale_min_eff}, {scale_max_eff}]) …")

    scales, Fqs, hq = mfdfa(
        ret, Q_HURST,
        scale_min=scale_min_eff, scale_max=scale_max_eff,
        n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP,
    )
    tau_q, alpha, D_alpha = multifractal_spectrum(Q_HURST, hq)

    # ── Compute λ_MF and spectral D_H ────────────────────────────────────────
    lam_dict = lyapunov_mf(ret, Q_HURST, hq, alpha, D_alpha)
    print(f"  λ_MF        = {lam_dict['lam_mf']:.5f}  "
          f"[H₂={lam_dict['H2']:.3f}, σ={lam_dict['sigma']:.4f}, "
          f"Δα={lam_dict['delta_alpha']:.3f}]")
    print(f"  D_H         = {lam_dict['D_H_spectral']:.5f}  "
          f"[= max f(α), spectral — no box-counting]")

    # ── Plot MF-DFA ───────────────────────────────────────────────────────────
    plot_mfdfa(ticker, ret, scales, Fqs, hq, Q_HURST,
               tau_q, alpha, D_alpha, lam_dict, mfd_dir)

    # ── Cumulative characteristic analysis ────────────────────────────────────
    print("  → Cumulative characteristic analysis …")
    print("    (box-counting used only for D_H_geom(q) — induced complexity)")
    plot_characteristic(ticker, ret, lam_dict, char_dir)

print(f"\n{'='*60}")
print(f"Done.  All results saved in:  {OUT_DIR}/")


Ticker: AAPL
  N = 4062 returns
  → MF-DFA  (scales [20, 800]) …
  λ_MF        = 0.01416  [H₂=0.536, σ=0.0177, Δα=0.491]
  D_H         = 1.01576  [= max f(α), spectral — no box-counting]
    saved → fractal_structure_results\AAPL\mfdfa\AAPL_mfdfa.png
  → Cumulative characteristic analysis …
    (box-counting used only for D_H_geom(q) — induced complexity)
    saved → fractal_structure_results\AAPL\characteristic_curves\AAPL_characteristic.png

Ticker: MSFT
  N = 4062 returns
  → MF-DFA  (scales [20, 800]) …
  λ_MF        = 0.01091  [H₂=0.454, σ=0.0162, Δα=0.488]
  D_H         = 1.00599  [= max f(α), spectral — no box-counting]
    saved → fractal_structure_results\MSFT\mfdfa\MSFT_mfdfa.png
  → Cumulative characteristic analysis …
    (box-counting used only for D_H_geom(q) — induced complexity)
    saved → fractal_structure_results\MSFT\characteristic_curves\MSFT_characteristic.png

Ticker: GOOG
  N = 4062 returns
  → MF-DFA  (scales [20, 800]) …
  λ_MF        = 0.01112  [H₂=0.474, σ=

In [9]:
"""
Fractal Structure Analysis of Financial Time Series
=====================================================

Three analysis modules per ticker:

  1. CUMULATIVE CHARACTERISTIC ANALYSIS (first order)
     - Characteristic curves F_q from returns
     
  2. MF-DFA (standard multifractal analysis)
  
  3. SECOND-ORDER CHARACTERISTIC ANALYSIS (NEW)
     - Treat each F_q curve as a complex time series
     - Compute ITS characteristic curves G_{q2}(F_q)
     - Analyze fractal structure of the characteristic curves themselves
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.signal import savgol_filter
from scipy import stats
import os
import warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
TICKERS   = ["AAPL", "MSFT", "GOOG", "AMZN", "NVDA", "JPM", "XOM", "BTC-USD"]
START     = "2010-01-01"
END       = "2026-03-01"
OUT_DIR   = "fractal_structure_results"

# q values for cumulative characteristic (wavenumber analogue)
Q_CHAR         = np.array([0, 1, 5, 10, 20, 60, 120, 200, 300], dtype=float)

# Second-order q values (for analyzing the characteristic curves themselves)
Q_SECOND_ORDER = np.array([0, 1, 2, 5, 10, 20], dtype=float)

# q values for MF-DFA
Q_HURST = np.arange(-30, 31, 5, dtype=float)

# MF-DFA scale parameters
SCALE_MIN = 20
SCALE_MAX = 800
N_SCALES  = 30
POLY_ORD  = 1
OVERLAP   = 0.75

# Palette for F_q(s) lines
CMAP_LINES = [
    "#c0392b", "#2980b9", "#27ae60", "#8e44ad",
    "#d35400", "#16a085", "#2c3e50", "#e67e22",
    "#1abc9c", "#e74c3c", "#3498db",
]

os.makedirs(OUT_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# STYLE
# ─────────────────────────────────────────────────────────────────────────────
PLT_STYLE = dict(
    fig_fc    = "#ffffff",
    ax_fc     = "#ffffff",
    text      = "#1a1a1a",
    grid      = "#e5e5e5",
    spine     = "#cccccc",
    accent    = "#1a1a2e",
    line_main = "#1f4e8c",
    line_alt  = "#c0392b",
    line_grn  = "#1a7a4a",
)

def _style(fig, axes_flat):
    fig.patch.set_facecolor(PLT_STYLE["fig_fc"])
    for ax in axes_flat:
        ax.set_facecolor(PLT_STYLE["ax_fc"])
        ax.tick_params(colors=PLT_STYLE["text"], which="both",
                       direction="in", top=True, right=True, labelsize=8)
        ax.xaxis.label.set_color(PLT_STYLE["text"])
        ax.yaxis.label.set_color(PLT_STYLE["text"])
        ax.title.set_color(PLT_STYLE["accent"])
        for sp in ax.spines.values():
            sp.set_edgecolor(PLT_STYLE["spine"])
        ax.grid(True, color=PLT_STYLE["grid"], linewidth=0.5,
                linestyle="--", alpha=0.9)

def _legend(ax, **kw):
    return ax.legend(framealpha=0.9, facecolor="white",
                     edgecolor=PLT_STYLE["spine"],
                     labelcolor=PLT_STYLE["text"], **kw)


# ─────────────────────────────────────────────────────────────────────────────
# DIRECTORIES
# ─────────────────────────────────────────────────────────────────────────────
def get_dirs(ticker):
    t    = ticker.replace("=", "_")
    base = os.path.join(OUT_DIR, t)
    char = os.path.join(base, "characteristic_curves")
    mfd  = os.path.join(base, "mfdfa")
    second = os.path.join(base, "second_order")
    os.makedirs(char, exist_ok=True)
    os.makedirs(mfd,  exist_ok=True)
    os.makedirs(second, exist_ok=True)
    return char, mfd, second


# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────
def get_data(ticker):
    df = yf.download(ticker, start=START, end=END, progress=False)
    if df is None or len(df) == 0:
        return None, None
    try:
        if hasattr(df.columns, "levels"):
            px = df["Close"][ticker].dropna().values
        else:
            px = df["Close"].dropna().values
    except Exception:
        px = df["Close"].dropna().values

    px = px.ravel().astype(float)
    px = px[px > 0]
    if len(px) < 200:
        return None, None

    logp = np.log(px)
    ret  = np.diff(logp)
    ret  = ret[np.isfinite(ret)]
    if len(ret) < 200:
        return None, None
    return logp, ret


# ─────────────────────────────────────────────────────────────────────────────
# MF-DFA
# ─────────────────────────────────────────────────────────────────────────────
def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)


def mfdfa(x, q_vals, scale_min=SCALE_MIN, scale_max=SCALE_MAX,
          n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP):
    """MF-DFA on log-returns with overlapping segments."""
    x  = x[np.isfinite(x)]
    Y  = np.cumsum(x - x.mean())
    N  = len(Y)

    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    Fqs = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            continue
        t   = np.arange(s)
        rms = []
        for st in starts:
            seg  = Y[st: st + s]
            coef = np.polyfit(t, seg, poly_ord)
            r_v  = np.sqrt(np.mean((seg - np.polyval(coef, t)) ** 2))
            if r_v > 0:
                rms.append(r_v)
        rms = np.asarray(rms)
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms ** 2)))
            else:
                Fqs[j, i] = (np.mean(rms ** q)) ** (1.0 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]

    return scales, Fqs, hq


def multifractal_spectrum(q_vals, hq):
    """τ(q), α(q), f(α) from H(q)."""
    tau_q = q_vals * hq - 1.0
    wl    = min(5, len(tau_q) if len(tau_q) % 2 == 1 else len(tau_q) - 1)
    tau_s = (savgol_filter(tau_q, window_length=wl, polyorder=2)
             if len(tau_q) >= 5 else tau_q)
    alpha   = np.gradient(tau_s, q_vals)
    D_alpha = q_vals * alpha - tau_s
    return tau_q, alpha, D_alpha


def hausdorff_from_spectrum(D_alpha):
    """True Hausdorff dimension from multifractal spectrum."""
    valid = np.isfinite(D_alpha)
    if valid.sum() < 2:
        return 1.0
    return float(np.clip(np.nanmax(D_alpha[valid]), 1.0, 2.0))


def hausdorff_boxcount(F, n_eps=15):
    """Box-counting dimension of a complex curve."""
    x, y = F.real, F.imag
    if len(x) < 10:
        return 1.0
    eps    = np.logspace(-1.0, -2.5, n_eps)
    counts = [len(set(zip((x / e).astype(int), (y / e).astype(int))))
              for e in eps]
    counts = np.array(counts, dtype=float)
    valid  = counts > 0
    if valid.sum() < 3:
        return 1.0
    slope, _ = np.polyfit(np.log(eps[valid]), np.log(counts[valid]), 1)
    return float(np.clip(-slope, 1.0, 2.0))


def lyapunov_mf(ret, q_vals, hq, alpha, D_alpha):
    """Estimate Lyapunov exponent from multifractal spectrum."""
    sigma = float(np.std(ret))

    q2_idx = np.argmin(np.abs(q_vals - 2.0))
    H2     = float(hq[q2_idx]) if np.isfinite(hq[q2_idx]) else 0.5

    valid_s = np.isfinite(alpha) & np.isfinite(D_alpha)
    if valid_s.sum() >= 2:
        delta_alpha = float(alpha[valid_s].max() - alpha[valid_s].min())
        alpha0      = float(alpha[np.argmin(np.abs(q_vals))])
    else:
        delta_alpha = 0.0
        alpha0      = H2

    lam_mf = H2 * sigma * (1.0 + delta_alpha)
    lam_mf = float(np.clip(lam_mf, 1e-6, None))

    D_H_spectral = hausdorff_from_spectrum(D_alpha)

    return dict(
        lam_mf       = lam_mf,
        H2           = H2,
        sigma        = sigma,
        delta_alpha  = delta_alpha,
        alpha0       = alpha0,
        D_H_spectral = D_H_spectral,
    )


# ─────────────────────────────────────────────────────────────────────────────
# FIRST-ORDER CHARACTERISTIC CURVE
# ─────────────────────────────────────────────────────────────────────────────
def cumulative_characteristic(r, q, n_grid=2000):
    """Compute F_q = sum(e^{i*q*r}) for returns r."""
    r = np.asarray(r, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 10:
        return np.linspace(0, 1, n_grid) + 0j

    if len(r) > n_grid:
        idx = np.linspace(0, len(r) - 1, n_grid).astype(int)
        r_used = r[idx]
    else:
        r_used = r

    F = np.cumsum(np.exp(1j * q * r_used))

    d = F[-1] - F[0]
    if np.abs(d) < 1e-12:
        return F / (np.max(np.abs(F)) + 1e-12)
    return (F - F[0]) / d


# ─────────────────────────────────────────────────────────────────────────────
# SECOND-ORDER CHARACTERISTIC ANALYSIS
# Apply the same characteristic transform to the characteristic curves themselves
# ─────────────────────────────────────────────────────────────────────────────
def second_order_characteristic(curve, q2):
    """
    Compute characteristic curve of a characteristic curve.
    Treats the complex curve F_q as a complex-valued time series.
    """
    curve = np.asarray(curve)
    if len(curve) < 10:
        return np.array([0+0j])
    
    # curve is already complex (1D array of complex numbers)
    # Compute incremental steps as complex differences
    steps = np.diff(curve)  # This gives complex numbers directly
    
    # Apply characteristic transform to the steps of the curve
    G = np.cumsum(np.exp(1j * q2 * steps))
    
    # Normalize
    if len(G) > 1:
        d = G[-1] - G[0]
        if np.abs(d) < 1e-12:
            G = G / (np.max(np.abs(G)) + 1e-12)
        else:
            G = (G - G[0]) / d
    
    return G
def analyze_second_order(ticker, ret, char_dir, second_dir):
    """
    Analyze the fractal structure of the characteristic curves themselves.
    For each first-order F_q, compute its second-order characteristic curves G_{q2}.
    """
    print("\n  → SECOND-ORDER CHARACTERISTIC ANALYSIS")
    print("    (applying the same transform to the characteristic curves themselves)")
    
    # First, compute all first-order characteristic curves
    first_order_curves = {}
    print("    Computing first-order characteristic curves...")
    for q in Q_CHAR:
        F = cumulative_characteristic(ret, q)
        first_order_curves[q] = F
        print(f"      q={int(q)}: curve with {len(F)} points")
    
    # For each first-order q, compute second-order curves for different q2
    second_order_results = {}
    
    # Select a subset of first-order q for clarity in plots
    q_subset = [0, 5, 20, 60, 200]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    fig.suptitle(
        f"Second-Order Characteristic Analysis  ·  {ticker}\n"
        r"Characteristic curves of the characteristic curves $G_{q2}(F_q)$",
        fontsize=12, fontweight="bold", color=PLT_STYLE["accent"], y=1.02,
    )
    _style(fig, axes.ravel())
    
    # Panel (a): First-order curves (reference)
    ax = axes[0, 0]
    colors_first = plt.cm.viridis(np.linspace(0, 0.9, len(q_subset)))
    for idx, q in enumerate(q_subset):
        F = first_order_curves[q]
        ax.plot(F.real, F.imag, '-', color=colors_first[idx], 
               linewidth=1.5, alpha=0.8, label=f'F_{{{int(q)}}}')
    ax.set_xlabel('Re $F_q$', fontsize=10)
    ax.set_ylabel('Im $F_q$', fontsize=10)
    ax.set_title('(a) First-order characteristic curves $F_q$', fontsize=10)
    ax.set_aspect('equal')
    _legend(ax, fontsize=7, ncol=2)
    
    # Panel (b): Second-order curves for a fixed q (showing how G_{q2} varies with q2)
    ax = axes[0, 1]
    fixed_q = 20  # Pick a representative first-order q
    F_fixed = first_order_curves[fixed_q]
    
    colors_second = plt.cm.plasma(np.linspace(0, 0.9, len(Q_SECOND_ORDER)))
    for idx, q2 in enumerate(Q_SECOND_ORDER):
        G = second_order_characteristic(F_fixed, q2)
        ax.plot(G.real, G.imag, '-', color=colors_second[idx], 
               linewidth=1.5, alpha=0.8, label=f'$G_{{{int(q2)}}}(F_{{{fixed_q}}})$')
        # Compute fractal dimension of this second-order curve
        D_second = hausdorff_boxcount(G)
        print(f"      q={fixed_q}, q2={int(q2)}: D_H(G) = {D_second:.4f}")
    
    ax.set_xlabel('Re $G_{q2}$', fontsize=10)
    ax.set_ylabel('Im $G_{q2}$', fontsize=10)
    ax.set_title(f'(b) Second-order curves $G_{{q2}}(F_{{{fixed_q}}})$', fontsize=10)
    ax.set_aspect('equal')
    _legend(ax, fontsize=6, ncol=1, loc='center right')
    
    # Panel (c): Fractal dimension of second-order curves vs q2 for multiple first-order q
    ax = axes[1, 0]
    
    for idx, q in enumerate(q_subset):
        F = first_order_curves[q]
        D_vals = []
        for q2 in Q_SECOND_ORDER:
            G = second_order_characteristic(F, q2)
            D = hausdorff_boxcount(G)
            D_vals.append(D)
        ax.plot(Q_SECOND_ORDER, D_vals, 'o-', color=colors_first[idx],
               linewidth=1.5, markersize=5, label=f'F_{{{int(q)}}}')
    
    ax.set_xlabel('Second-order wavenumber $q_2$', fontsize=10)
    ax.set_ylabel('Fractal dimension $D_H(G_{q2})$', fontsize=10)
    ax.set_title('(c) Dimension of second-order curves vs $q_2$', fontsize=10)
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='D=1 (line)')
    ax.axhline(2.0, color='gray', linestyle='--', alpha=0.5, label='D=2 (area)')
    _legend(ax, fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)
    
    # Panel (d): D_H(G) vs D_H(F) - relationship between first and second order dimensions
    ax = axes[1, 1]
    
    # For a fixed q2, plot second-order dimension vs first-order dimension
    fixed_q2 = 5
    first_dims = []
    second_dims = []
    
    for q in q_subset:
        F = first_order_curves[q]
        D_first = hausdorff_boxcount(F)
        G = second_order_characteristic(F, fixed_q2)
        D_second = hausdorff_boxcount(G)
        first_dims.append(D_first)
        second_dims.append(D_second)
        ax.annotate(f'q={int(q)}', xy=(D_first, D_second), xytext=(3, 3),
                   textcoords='offset points', fontsize=8)
    
    ax.scatter(first_dims, second_dims, s=80, color=PLT_STYLE["line_main"],
              edgecolors='black', linewidth=1, alpha=0.7, zorder=5)
    
    # Fit line
    if len(first_dims) >= 3:
        slope, intercept, r_value, _, _ = stats.linregress(first_dims, second_dims)
        x_line = np.linspace(min(first_dims), max(first_dims), 100)
        ax.plot(x_line, intercept + slope * x_line, '--',
               color=PLT_STYLE["line_alt"], linewidth=1.5,
               label=f'D_H(G) = {intercept:.3f} + {slope:.3f}·D_H(F)\nR²={r_value**2:.3f}')
    
    ax.set_xlabel('Fractal dimension of $F_q$', fontsize=10)
    ax.set_ylabel(f'Fractal dimension of $G_{{{fixed_q2}}}(F_q)$', fontsize=10)
    ax.set_title(f'(d) First-order vs second-order dimension ($q_2={fixed_q2}$)', fontsize=10)
    _legend(ax, fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    fname = os.path.join(second_dir, f"{ticker.replace('=','_')}_second_order.png")
    plt.savefig(fname, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"    saved → {fname}")
    
    # Also create individual plots for each first-order q
    for q in q_subset:
        fig, ax = plt.subplots(figsize=(10, 8))
        F = first_order_curves[q]
        
        colors_second = plt.cm.plasma(np.linspace(0, 0.9, len(Q_SECOND_ORDER)))
        for idx, q2 in enumerate(Q_SECOND_ORDER):
            G = second_order_characteristic(F, q2)
            D = hausdorff_boxcount(G)
            ax.plot(G.real, G.imag, '-', color=colors_second[idx],
                   linewidth=1.5, alpha=0.8, 
                   label=f'$G_{{{int(q2)}}}$, D={D:.3f}')
        
        ax.set_xlabel('Re $G_{q2}$', fontsize=12)
        ax.set_ylabel('Im $G_{q2}$', fontsize=12)
        ax.set_title(f'{ticker}: Second-order curves $G_{{q2}}(F_{{{int(q)}}})$', 
                    fontsize=14, fontweight='bold')
        ax.set_aspect('equal')
        _legend(ax, fontsize=8, ncol=2)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        fname = os.path.join(second_dir, f"{ticker.replace('=','_')}_second_order_q{q}.png")
        plt.savefig(fname, dpi=150, bbox_inches="tight", facecolor="white")
        plt.close(fig)
    
    return second_order_results

# ─────────────────────────────────────────────────────────────────────────────
# PLOT A — FIRST-ORDER CUMULATIVE CHARACTERISTIC
# ─────────────────────────────────────────────────────────────────────────────
def plot_characteristic(ticker, ret, lam_dict, char_dir):
    """4-panel figure for first-order cumulative characteristic analysis."""
    DH_geom = np.array([hausdorff_boxcount(cumulative_characteristic(ret, q))
                        for q in Q_CHAR])
    q2      = Q_CHAR ** 2
    valid_m = np.isfinite(DH_geom) & (q2 > 0)

    slope_fit, intercept_fit = (np.polyfit(q2[valid_m], DH_geom[valid_m], 1)
                                if valid_m.sum() >= 2 else (np.nan, 1.0))

    lam_mf       = lam_dict["lam_mf"]
    H2           = lam_dict["H2"]
    sigma        = lam_dict["sigma"]
    delta_alpha  = lam_dict["delta_alpha"]
    D_H_spectral = lam_dict["D_H_spectral"]

    D_implied = slope_fit * lam_mf if np.isfinite(slope_fit) else np.nan

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"First-Order Characteristic Analysis  ·  {ticker}",
        fontsize=13, fontweight="bold", color=PLT_STYLE["accent"], y=1.01,
    )
    _style(fig, axes.ravel())

    # (a) curves in ℂ
    ax = axes[0, 0]
    for i, q in enumerate(Q_CHAR):
        c = plt.cm.magma_r(i / len(Q_CHAR))
        F  = cumulative_characteristic(ret, q)
        lw = 1.0 if q > 20 else 1.6
        ax.plot(F.real, F.imag, linewidth=lw, color=c,
                label=f"$q={int(q)}$", alpha=0.88)
    ax.set_xlabel("Re $F_q$", fontsize=10)
    ax.set_ylabel("Im $F_q$", fontsize=10)
    ax.set_title("(a) First-order characteristic curves $F_q$", fontsize=10)
    _legend(ax, fontsize=8)
    ax.set_aspect("equal", adjustable="datalim")
    ax.text(0.03, 0.97,
            fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="left", va="top", fontsize=8,
            color=PLT_STYLE["line_grn"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    # (b) D_H_geom(q) vs q²
    ax = axes[0, 1]
    ax.scatter(q2, DH_geom, color=PLT_STYLE["line_main"], s=60, zorder=5,
               edgecolors="#aaaaaa", linewidths=0.5,
               label=r"$D_H^{\rm geom}(q)$  (box-count of $F_q$)")
    if np.isfinite(slope_fit):
        q2_line = np.linspace(0, q2.max() * 1.06, 200)
        ax.plot(q2_line, intercept_fit + slope_fit * q2_line,
                color=PLT_STYLE["line_alt"], linewidth=1.5, linestyle="--",
                label=fr"fit  slope$={slope_fit:.2e}$")
        if np.isfinite(D_implied):
            ax.plot(q2_line, 1.0 + (D_implied / lam_mf) * q2_line,
                    color=PLT_STYLE["line_grn"], linewidth=1.2, linestyle=":",
                    label=fr"$\hat{{D}}/\lambda_{{MF}}={D_implied/lam_mf:.2e}$")
    ax.axhline(D_H_spectral, color=PLT_STYLE["line_grn"], linewidth=1.2,
               linestyle="-.",
               label=fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$")
    ax.axhline(1.0, color="#aaaaaa", linewidth=0.8, linestyle=":")
    ax.set_xlabel("$q^2$",    fontsize=10)
    ax.set_ylabel(r"$D_H^{\rm geom}(q)$", fontsize=10)
    ax.set_title(r"(b) $D_H^{\rm geom}(q)$ vs $q^2$", fontsize=9)
    _legend(ax, fontsize=7)
    ax.text(0.97, 0.12,
            fr"$\lambda_{{MF}} = {lam_mf:.4f}$",
            transform=ax.transAxes, ha="right", fontsize=9,
            color=PLT_STYLE["line_main"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    # (c) λ_MF component breakdown
    ax = axes[1, 0]
    labels  = ["$H(q{=}2)$", r"$\sigma$", r"$1+\Delta\alpha$", r"$\lambda_{MF}$"]
    values  = [H2, sigma, 1.0 + delta_alpha, lam_mf]
    colours = [PLT_STYLE["line_main"], PLT_STYLE["line_grn"],
               "#e67e22", PLT_STYLE["line_alt"]]
    bars = ax.bar(labels, values, color=colours, edgecolor="#cccccc",
                  linewidth=0.8, alpha=0.85, width=0.55)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values) * 0.02,
                f"{val:.4f}", ha="center", va="bottom",
                fontsize=8, color=PLT_STYLE["text"])
    ax.set_ylabel("Value", fontsize=10)
    ax.set_title(
        r"(c)  $\lambda_{MF} = H(q{=}2)\cdot\sigma\cdot(1+\Delta\alpha)$",
        fontsize=10)
    ax.set_ylim(0, max(values) * 1.18)
    ax.text(0.97, 0.97,
            fr"$\lambda_{{MF}} = {H2:.3f}\times{sigma:.4f}\times{1+delta_alpha:.3f}$"
            f"\n$= {lam_mf:.5f}$"
            f"\n$D_H = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            color=PLT_STYLE["text"],
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    # (d) Implied diffusion coefficient
    ax = axes[1, 1]
    dh_excess = DH_geom - 1.0
    q2_scaled = q2 / lam_mf
    ax.scatter(q2_scaled[valid_m], dh_excess[valid_m],
               color=PLT_STYLE["line_main"], s=60, zorder=5,
               edgecolors="#aaaaaa", linewidths=0.5,
               label=r"$(D_H^{\rm geom} - 1)$ vs $q^2/\lambda_{MF}$")
    if np.isfinite(D_implied) and valid_m.sum() >= 2:
        x_line = np.linspace(0, q2_scaled[valid_m].max() * 1.06, 200)
        ax.plot(x_line, D_implied * x_line,
                color=PLT_STYLE["line_alt"], linewidth=1.5, linestyle="--",
                label=fr"slope $= \hat{{D}} = {D_implied:.4f}$")
    ax.axhline(0, color="#aaaaaa", linewidth=0.8, linestyle=":")
    ax.axhline(D_H_spectral - 1.0, color=PLT_STYLE["line_grn"],
               linewidth=1.0, linestyle="-.",
               label=fr"$D_H - 1 = {D_H_spectral-1:.4f}$  (spectral)")
    ax.set_xlabel(r"$q^2 / \lambda_{MF}$",  fontsize=10)
    ax.set_ylabel(r"$D_H^{\rm geom} - 1$",  fontsize=10)
    ax.set_title(r"(d) Implied diffusion coefficient $\hat{\mathcal{D}}$",
                 fontsize=10)
    _legend(ax, fontsize=7)
    if np.isfinite(D_implied):
        ax.text(0.97, 0.12,
                fr"$\hat{{\mathcal{{D}}}} = {D_implied:.5f}$",
                transform=ax.transAxes, ha="right", fontsize=9,
                color=PLT_STYLE["line_alt"],
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                          edgecolor=PLT_STYLE["spine"], alpha=0.85))

    plt.tight_layout()
    fname = os.path.join(char_dir, f"{ticker.replace('=','_')}_characteristic.png")
    plt.savefig(fname, dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"    saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT B — MF-DFA
# ─────────────────────────────────────────────────────────────────────────────
def plot_mfdfa(ticker, ret, scales, Fqs, hq, q_vals,
               tau_q, alpha, D_alpha, lam_dict, mfd_dir):
    valid   = np.isfinite(hq)
    valid_s = np.isfinite(alpha) & np.isfinite(D_alpha)
    colors  = (CMAP_LINES * 5)[: len(q_vals)]

    lam_mf       = lam_dict["lam_mf"]
    H2           = lam_dict["H2"]
    delta_alpha  = lam_dict["delta_alpha"]
    alpha0       = lam_dict["alpha0"]
    D_H_spectral = lam_dict["D_H_spectral"]

    fig, axs = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(
        f"Multifractal DFA  ·  {ticker}  ·  log-returns",
        fontsize=13, fontweight="bold", color=PLT_STYLE["accent"], y=1.01,
    )
    _style(fig, axs.ravel())

    ax = axs[0, 0]
    ax.plot(ret, color=PLT_STYLE["line_main"], linewidth=0.55, alpha=0.9)
    ax.set_xlabel("$t$  (trading days)", fontsize=10)
    ax.set_ylabel("Log-return",           fontsize=10)
    ax.set_title("(a) Log-return series", fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3g"))

    ax = axs[0, 1]
    legend_handles = []
    for idx_q, (q, c) in enumerate(zip(q_vals, colors)):
        row  = Fqs[idx_q]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() < 2:
            continue
        h, = ax.loglog(scales[mask], row[mask], "o-",
                       color=c, markersize=2.8, linewidth=0.85, alpha=0.8)
        legend_handles.append((h, f"$q={int(q)}$"))
    ax.set_xlabel("Scale $s$", fontsize=10)
    ax.set_ylabel("$F_q(s)$",  fontsize=10)
    ax.set_title("(b) Fluctuation functions $F_q(s)$", fontsize=10)
    step  = max(1, len(legend_handles) // 7)
    shown = legend_handles[::step]
    if shown:
        _legend(ax, handles=[h for h, _ in shown],
                labels=[l for _, l in shown],
                fontsize=7, ncol=1, loc="best",
                handlelength=1.2, handletextpad=0.4)

    ax = axs[0, 2]
    ax.plot(q_vals[valid], hq[valid], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    ax.axhline(0.5, color="#aaaaaa", lw=0.9, ls="--", alpha=0.9, label="$H=0.5$")
    ax.axhline(1.0, color=PLT_STYLE["line_alt"], lw=0.9, ls=":", alpha=0.9,
               label="$H=1.0$")
    q2_idx = np.argmin(np.abs(q_vals - 2.0))
    if np.isfinite(hq[q2_idx]):
        ax.plot(q_vals[q2_idx], hq[q2_idx], "o",
                color=PLT_STYLE["line_alt"], markersize=7, zorder=6,
                label=f"$H(q{{=}}2)={H2:.3f}$")
    ax.set_xlabel("$q$",    fontsize=10)
    ax.set_ylabel("$H(q)$", fontsize=10)
    ax.set_title("(c) Generalised Hurst exponent $H(q)$", fontsize=10)
    _legend(ax, fontsize=8)
    ax.text(0.03, 0.05,
            fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="left", va="bottom", fontsize=8,
            color=PLT_STYLE["line_grn"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    ax = axs[1, 0]
    ax.plot(q_vals[valid], tau_q[valid], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    ax.axhline(0, color="#aaaaaa", lw=0.8, ls="--", alpha=0.8)
    ax.set_xlabel("$q$",        fontsize=10)
    ax.set_ylabel(r"$\tau(q)$", fontsize=10)
    ax.set_title(r"(d) Scaling exponent $\tau(q)$", fontsize=10)

    ax = axs[1, 1]
    ax.plot(q_vals[valid_s], alpha[valid_s], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    q0_idx = np.argmin(np.abs(q_vals))
    if np.isfinite(alpha[q0_idx]):
        ax.plot(q_vals[q0_idx], alpha[q0_idx], "o",
                color=PLT_STYLE["line_alt"], markersize=7, zorder=6,
                label=fr"$\alpha_0={alpha0:.3f}$")
    ax.set_xlabel("$q$",          fontsize=10)
    ax.set_ylabel(r"$\alpha(q)$", fontsize=10)
    ax.set_title(r"(e) Singularity strength $\alpha(q)$", fontsize=10)
    _legend(ax, fontsize=8)

    ax = axs[1, 2]
    ax.plot(alpha[valid_s], D_alpha[valid_s], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    if np.isfinite(alpha[q0_idx]) and np.isfinite(D_alpha[q0_idx]):
        ax.plot(alpha[q0_idx], D_alpha[q0_idx], "o",
                color=PLT_STYLE["line_alt"], markersize=7, zorder=6,
                label=f"$q=0$  ($\\alpha_0={alpha0:.3f}$)")
    if valid_s.sum() >= 2:
        peak_idx   = np.nanargmax(D_alpha[valid_s])
        alpha_peak = alpha[valid_s][peak_idx]
        f_peak     = D_alpha[valid_s][peak_idx]
        ax.plot(alpha_peak, f_peak, "^",
                color=PLT_STYLE["line_grn"], markersize=9, zorder=7,
                label=fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$")
        ax.annotate(fr"$D_H={D_H_spectral:.4f}$",
                    xy=(alpha_peak, f_peak),
                    xytext=(alpha_peak + 0.02, f_peak - 0.06),
                    fontsize=8, color=PLT_STYLE["line_grn"],
                    arrowprops=dict(arrowstyle="->", color=PLT_STYLE["line_grn"],
                                   lw=0.9))
        a_min  = alpha[valid_s].min()
        a_max  = alpha[valid_s].max()
        y_span = D_alpha[valid_s].min() - 0.05
        ax.annotate("", xy=(a_max, y_span), xytext=(a_min, y_span),
                    arrowprops=dict(arrowstyle="<->", color="#888888", lw=1.2))
        ax.text((a_min + a_max) / 2, y_span - 0.04,
                fr"$\Delta\alpha={delta_alpha:.3f}$",
                ha="center", fontsize=8, color=PLT_STYLE["text"])
    ax.set_xlabel(r"$\alpha$",    fontsize=10)
    ax.set_ylabel(r"$f(\alpha)$", fontsize=10)
    ax.set_title(r"(f) Multifractal spectrum $f(\alpha)$", fontsize=9)
    _legend(ax, fontsize=8)
    ax.text(0.97, 0.30,
            fr"$\lambda_{{MF}} = {lam_mf:.5f}$"
            "\n"
            fr"$= H_2\cdot\sigma\cdot(1+\Delta\alpha)$"
            "\n"
            fr"$D_H = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            color=PLT_STYLE["text"],
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.9))

    plt.tight_layout()
    fname = os.path.join(mfd_dir, f"{ticker.replace('=','_')}_mfdfa.png")
    plt.savefig(fname, dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"    saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Ticker: {ticker}")

    char_dir, mfd_dir, second_dir = get_dirs(ticker)
    logp, ret = get_data(ticker)

    if ret is None:
        print("  Skipping — insufficient data.")
        continue

    N = len(ret)
    print(f"  N = {N} returns")

    # MF-DFA
    scale_max_eff = min(SCALE_MAX, N // 4)
    scale_min_eff = max(SCALE_MIN, POLY_ORD + 2)
    print(f"  → MF-DFA  (scales [{scale_min_eff}, {scale_max_eff}]) …")

    scales, Fqs, hq = mfdfa(
        ret, Q_HURST,
        scale_min=scale_min_eff, scale_max=scale_max_eff,
        n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP,
    )
    tau_q, alpha, D_alpha = multifractal_spectrum(Q_HURST, hq)

    lam_dict = lyapunov_mf(ret, Q_HURST, hq, alpha, D_alpha)
    print(f"  λ_MF        = {lam_dict['lam_mf']:.5f}")
    print(f"  D_H         = {lam_dict['D_H_spectral']:.5f}")

    # First-order characteristic analysis
    print("  → First-order characteristic analysis …")
    plot_characteristic(ticker, ret, lam_dict, char_dir)
    
    # MF-DFA plot
    plot_mfdfa(ticker, ret, scales, Fqs, hq, Q_HURST,
               tau_q, alpha, D_alpha, lam_dict, mfd_dir)
    
    # SECOND-ORDER characteristic analysis (NEW)
    # This applies the same transform to the characteristic curves themselves!
    analyze_second_order(ticker, ret, char_dir, second_dir)

print(f"\n{'='*60}")
print(f"Done.  All results saved in:  {OUT_DIR}/")


Ticker: AAPL
  N = 4062 returns
  → MF-DFA  (scales [20, 800]) …
  λ_MF        = 0.01416
  D_H         = 1.01576
  → First-order characteristic analysis …
    saved → fractal_structure_results\AAPL\characteristic_curves\AAPL_characteristic.png
    saved → fractal_structure_results\AAPL\mfdfa\AAPL_mfdfa.png

  → SECOND-ORDER CHARACTERISTIC ANALYSIS
    (applying the same transform to the characteristic curves themselves)
    Computing first-order characteristic curves...
      q=0: curve with 2000 points
      q=1: curve with 2000 points
      q=5: curve with 2000 points
      q=10: curve with 2000 points
      q=20: curve with 2000 points
      q=60: curve with 2000 points
      q=120: curve with 2000 points
      q=200: curve with 2000 points
      q=300: curve with 2000 points
      q=20, q2=0: D_H(G) = 1.0000
      q=20, q2=1: D_H(G) = 1.0000
      q=20, q2=2: D_H(G) = 1.0000
      q=20, q2=5: D_H(G) = 1.0000
      q=20, q2=10: D_H(G) = 1.0000
      q=20, q2=20: D_H(G) = 1.0000
   